In [1]:
%matplotlib notebook

In [54]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import jax
import jax.numpy as jnp
import numpy as onp
import matplotlib.pyplot as plt

from msmjax.kernels import split_one_over_r_kernel, SofteningFunctionOneOverR
from msmjax.core.shortrange import make_compute_U_zero_with_neighborlist, \
    make_compute_U_and_f_zero_with_neighborlist
from msmjax.gridops_multidim import set_up_grids_all_levels
from msmjax.gridops_multidim import create_compute_U_oneplus_via_potential

import sys

sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmfornn.splines.nesting import compute_J_zeroplus
from msmfornn.gridtools import construct_grids_all_levels
from msmfornn.splines.coefficients import compute_coeffs_withtruncation
from msmfornn.grid_to_grid_mapping import \
    compute_kernel_stencils_all_gridlevels
from msmfornn.helpers.algoparam_choice import suggest_p, suggest_max_gridlevel_nonPBC


# Helper functions

In [3]:
def make_compute_U_zero_reference(kernels, periodic: bool, box_lengths=None):
    k_0 = kernels[0]
    sum_of_higher_kernels_at_zero = jnp.sum(
        jnp.asarray([k(0.0) for k in kernels[1:]])
    )
    
    if periodic and box_lengths is None:
        raise ValueError("`box_lengths` are required in periodic case.")
    
    def compute_U_zero_reference(positions, charges):
        R_ij = positions[:, jnp.newaxis, :] - positions
        if periodic:
            R_ij -= jnp.rint(R_ij / box_lengths) * box_lengths
        qi_qj = charges[:, jnp.newaxis] * charges
        indices_triu = jnp.triu_indices(positions.shape[0], k=1)
        r_ij_triu = jnp.linalg.norm(R_ij[indices_triu], axis=1)
        qi_qj_triu = qi_qj[indices_triu]
        
        pair_term = (qi_qj_triu * jax.vmap(k_0)(r_ij_triu)).sum()
        self_energy_term = 0.5 * jnp.diag(qi_qj).sum() * sum_of_higher_kernels_at_zero
    
        return pair_term - self_energy_term
    
    return compute_U_zero_reference

# Set up

In [4]:
# Basic geometry
AVG_NEIGHBOR_DISTANCE = 2.5
N_DIM = 3
PBCS = [False] * N_DIM

# MSM
N_LEVELS = 4
ALPHA = 4.0
P = 4
MU = 4
LEVEL_ONE_GRIDSPACING = AVG_NEIGHBOR_DISTANCE
LEVEL_ZERO_CUTOFF = ALPHA * LEVEL_ONE_GRIDSPACING

In [5]:
kernels = split_one_over_r_kernel(
        max_level=N_LEVELS,
        level_zero_cutoff=LEVEL_ZERO_CUTOFF,
        softening_function=SofteningFunctionOneOverR(P),
    )

In [6]:
rng = onp.random.default_rng(2489)

# Randomly drawn particle positions
n_particles = 2000
side_length = n_particles ** (1. / N_DIM) * AVG_NEIGHBOR_DISTANCE
box_lengths = jnp.array([side_length] * N_DIM)
pos = rng.uniform(low=onp.zeros_like(box_lengths), high=box_lengths, size=(n_particles, N_DIM))

# # Particles placed on a regular grid
# n_points_per_direction = 9
# n_particles = n_points_per_direction**N_DIM
# side_length = n_points_per_direction * AVG_NEIGHBOR_DISTANCE
# box_lengths = jnp.array([side_length] * N_DIM)
# mg = onp.meshgrid(*([onp.arange(n_points_per_direction)] * N_DIM), indexing="ij")
# pos = onp.stack([m.ravel() for m in mg], axis=1) * AVG_NEIGHBOR_DISTANCE

chg = rng.uniform(low=-1.0, high=1.0, size=n_particles)

pos = jnp.array(pos)
chg = jnp.array(chg)

box_lengths

Array([31.49802625, 31.49802625, 31.49802625], dtype=float64)

# Short-range

In [7]:
def make_wrapped_compute_U_zero(kernels, cutoff, box_lengths, pbcs, neighborlist_reference_positions):
    pbcs = jnp.asarray(pbcs)
    if not (jnp.all(pbcs) or jnp.all(~pbcs)):
        raise ValueError("Mixed boundary conditions currently not supported.")
    periodic = pbcs[0]
    
    # Too large box causes the neighbor list functions to throw strange errors.
    # But in the non-periodic case, the box size is actually irrelevant,
    # and we still get the correct result, and no error, by simply
    # specifying a fictitious small box.
    if not periodic:
        box_lengths = jnp.ones_like(box_lengths)
        
    neighbor_fun, compute_U_zero_with_neighborlist = make_compute_U_zero_with_neighborlist(
        kernels=kernels,
        cutoff=cutoff,
        box_lengths=box_lengths,
        pbcs=PBCS,
    )
    neighbor_list = neighbor_fun.allocate(neighborlist_reference_positions)
    
    def wrapped_compute_U_zero(positions, charges):
        updated_neighbor_list = neighbor_fun.update(positions, neighbor_list)
        return compute_U_zero_with_neighborlist(positions, charges, updated_neighbor_list.idx)
    
    return wrapped_compute_U_zero

In [8]:
compute_U_zero_reference = make_compute_U_zero_reference(
    kernels, periodic=PBCS[0], box_lengths=box_lengths
)
compute_U_zero_reference = jax.jit(compute_U_zero_reference)

wrapped_compute_U_zero_nbl = make_wrapped_compute_U_zero(
    kernels=kernels,
    cutoff=LEVEL_ZERO_CUTOFF,
    box_lengths=box_lengths,
    pbcs=PBCS,
    neighborlist_reference_positions=pos,
)
wrapped_compute_U_zero_nbl = jax.jit(wrapped_compute_U_zero_nbl)


/home/florian/anaconda3/envs/msmjax-dev/lib/python3.8/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int64 to dtype=int32. In future JAX releases this will result in an error.
  warnings.warn("scatter inputs have incompatible types: cannot safely cast "


In [9]:
U_zero_nbl = wrapped_compute_U_zero_nbl(pos, chg)
U_zero_reference = compute_U_zero_reference(pos, chg)

assert jnp.isclose(U_zero_nbl, U_zero_reference)

print(U_zero_reference)

-73.5451895651103


In [10]:
jax.device_put(pos)
jax.device_put(chg)

%timeit compute_U_zero_reference(pos, chg).block_until_ready()

2.95 ms ± 102 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [11]:
jax.device_put(pos)
jax.device_put(chg)

%timeit wrapped_compute_U_zero_nbl(pos, chg).block_until_ready()

502 µs ± 13.1 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [12]:
try:
    x, y, z, = pos[:, 0], pos[:, 1], pos[:, 2]

    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")
    ax.scatter(x.ravel(), y.ravel(), z.ravel(), c=chg, s=10)

    plt.show()
except ValueError:
    pass

<IPython.core.display.Javascript object>

In [13]:
def make_wrapped_compute_U_and_f_zero(kernels, cutoff, box_lengths, pbcs, neighborlist_reference_positions):
    pbcs = jnp.asarray(pbcs)
    if not (jnp.all(pbcs) or jnp.all(~pbcs)):
        raise ValueError("Mixed boundary conditions currently not supported.")
    periodic = pbcs[0]
    
    # Too large box causes the neighbor list functions to throw strange errors.
    # But in the non-periodic case, the box size is actually irrelevant,
    # and we still get the correct result, and no error, by simply
    # specifying a fictitious small box.
    if not periodic:
        box_lengths = jnp.ones_like(box_lengths)
        
    neighbor_fun, compute = make_compute_U_and_f_zero_with_neighborlist(
        kernels=kernels,
        cutoff=cutoff,
        box_lengths=box_lengths,
        pbcs=PBCS,
    )
    neighbor_list = neighbor_fun.allocate(neighborlist_reference_positions)
    
    def wrapped_compute(positions, charges):
        updated_neighbor_list = neighbor_fun.update(positions, neighbor_list)
        return compute(positions, charges, updated_neighbor_list.idx)
    
    return wrapped_compute

In [14]:
@jax.jit
def compute_U_and_f_zero_reference_grad(positions, charges):
    value, grad = jax.value_and_grad(compute_U_zero_reference, argnums=0)(positions, charges)
    return value, -grad

@jax.jit
def compute_U_and_f_zero_nbl_grad(positions, charges):
    value, grad = jax.value_and_grad(wrapped_compute_U_zero_nbl, argnums=0)(positions, charges)
    return value, -grad

wrapped_compute_U_and_f_zero_nbl = jax.jit(
    make_wrapped_compute_U_and_f_zero(
        kernels=kernels,
        cutoff=LEVEL_ZERO_CUTOFF,
        box_lengths=box_lengths,
        pbcs=PBCS,
        neighborlist_reference_positions=pos,
    )
)

/home/florian/anaconda3/envs/msmjax-dev/lib/python3.8/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int64 to dtype=int32. In future JAX releases this will result in an error.
  warnings.warn("scatter inputs have incompatible types: cannot safely cast "


In [15]:
compute_U_and_f_zero_reference_grad(pos, chg)

(Array(-73.54518957, dtype=float64),
 Array([[-0.4337324 ,  0.34794807,  0.13007648],
        [ 0.58681779,  0.30792036, -0.02517946],
        [-0.04745111, -0.03662173,  0.17531814],
        ...,
        [ 0.02962315,  0.16873253, -0.19158328],
        [-0.09549832, -0.42615689,  0.11051902],
        [ 0.17190791, -0.09445754, -0.04051903]], dtype=float64))

In [16]:
compute_U_and_f_zero_nbl_grad(pos, chg)

(Array(-73.54518957, dtype=float64),
 Array([[-0.4337324 ,  0.34794807,  0.13007648],
        [ 0.58681779,  0.30792036, -0.02517946],
        [-0.04745111, -0.03662173,  0.17531814],
        ...,
        [ 0.02962315,  0.16873253, -0.19158328],
        [-0.09549832, -0.42615689,  0.11051902],
        [ 0.17190791, -0.09445754, -0.04051903]], dtype=float64))

In [17]:
wrapped_compute_U_and_f_zero_nbl(pos, chg)

(Array(-73.54518957, dtype=float64),
 Array([[-0.4337324 ,  0.34794807,  0.13007648],
        [ 0.58681779,  0.30792036, -0.02517946],
        [-0.04745111, -0.03662173,  0.17531814],
        ...,
        [ 0.02962315,  0.16873253, -0.19158328],
        [-0.09549832, -0.42615689,  0.11051902],
        [ 0.17190791, -0.09445754, -0.04051903]], dtype=float64))

In [18]:
jax.device_put(pos)
jax.device_put(chg)

%timeit e, f = compute_U_and_f_zero_reference_grad(pos, chg)

7.68 ms ± 23.6 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [19]:
jax.device_put(pos)
jax.device_put(chg)

%timeit e, f = compute_U_and_f_zero_nbl_grad(pos, chg)

4.18 ms ± 11.3 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [20]:
jax.device_put(pos)
jax.device_put(chg)

%timeit e, f = wrapped_compute_U_and_f_zero_nbl(pos, chg)

11.9 ms ± 7.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


# Long-range

In [48]:
def wrapper_old_compute_J_zeroplus(p):
    return compute_J_zeroplus(p)

def wrapper_old_construct_kernel_stencils(
    kernels,
    box_lengths,
    level_one_gridspacing,
    level_zero_cutoff,
    n_levels,
    p,
    mu,
):
    omega, _ = compute_coeffs_withtruncation(p=p, mu=mu)
    omega_zeroplus = omega[len(omega) // 2 :]
    grids_oldmsm = construct_grids_all_levels(
        min_pos=onp.zeros_like(box_lengths),
        max_pos=box_lengths,
        p=p,
        level_one_gridspacing=level_one_gridspacing,
        max_gridlevel=n_levels,
    )
    kernel_stencils_nonnegative = compute_kernel_stencils_all_gridlevels(
        kernelfunctions=kernels,
        grids=grids_oldmsm,
        level_zero_cutoff=level_zero_cutoff,
        omega_zeroplus=omega_zeroplus,
    )
    kernel_stencils = [None]
    for stncl in kernel_stencils_nonnegative[1:]:
        pw = [(s - 1, 0) for s in stncl.shape]
        stncl_symm = jnp.pad(stncl, pad_width=pw, mode="reflect")
        kernel_stencils.append(stncl_symm)

    return kernel_stencils


In [49]:
def make_compute_U_oneplus(
    kernels,
    level_one_gridspacing,
    level_zero_cutoff,
    n_levels,
    box_lengths,
    pbcs,
    p,
    mu,
):
    n_dim = len(pbcs)

    J_zeroplus = wrapper_old_compute_J_zeroplus(p)
    grids = set_up_grids_all_levels(
        box_lengths=box_lengths,
        level_one_spacings=[level_one_gridspacing] * n_dim,
        pbcs=pbcs,
        n_levels=n_levels,
        p=p,
        J_zeroplus=J_zeroplus,
    )
    kernel_stencils = wrapper_old_construct_kernel_stencils(
        kernels=kernels,
        box_lengths=box_lengths,
        level_one_gridspacing=level_one_gridspacing,
        level_zero_cutoff=level_zero_cutoff,
        n_levels=n_levels,
        p=p,
        mu=mu,
    )

    calculate = create_compute_U_oneplus_via_potential(
        grids=grids, kernel_stencils=kernel_stencils
    )

    return calculate


In [57]:
calc_longrange_approx = make_compute_U_oneplus(
    kernels=kernels,
    level_one_gridspacing=LEVEL_ONE_GRIDSPACING,
    level_zero_cutoff=LEVEL_ZERO_CUTOFF,
    n_levels=N_LEVELS,  # TODO
    box_lengths=box_lengths,
    pbcs=PBCS,
    p=P,
    mu=MU,
)
calc_longrange_approx = jax.jit(calc_longrange_approx)

In [58]:
calc_longrange_approx(pos, chg)

Array(54.99171672, dtype=float64)

In [44]:
# J_zeroplus = compute_J_zeroplus(P)
# kernel_stencils = wrapper_old_construct_kernel_stencils(
#     box_lengths=box_lengths,
#     level_one_gridspacing=LEVEL_ONE_GRIDSPACING,
#     level_zero_cutoff=LEVEL_ZERO_CUTOFF,
#     n_levels=N_LEVELS,
#     p=P,
#     mu=MU,
# )
# grids = set_up_grids_all_levels(
#     box_lengths=box_lengths,
#     level_one_spacings=[LEVEL_ONE_GRIDSPACING] * N_DIM,
#     pbcs=PBCS,
#     n_levels=N_LEVELS,
#     p=P,
#     J_zeroplus=J_zeroplus,
# )

In [23]:
calc_longrange_e_approx = create_compute_U_oneplus_via_potential(grids=grids, kernel_stencils=kernel_stencils)
calc_longrange_e_approx = jax.jit(calc_longrange_e_approx)

In [24]:
calc_longrange_e_approx(pos, chg)

Array(57.89607963, dtype=float64)

In [25]:
jax.device_put(pos)
jax.device_put(chg)

%timeit calc_longrange_e_approx(pos, chg).block_until_ready()

7.6 ms ± 614 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


# Combined

In [26]:
@jax.jit
def calc_total_e_ref(positions, charges):
    R_ij = positions[:, jnp.newaxis, :] - positions
    qi_qj = charges[:, jnp.newaxis] * charges
    indices_triu = jnp.triu_indices(positions.shape[0], k=1)
    r_ij_triu = jnp.linalg.norm(R_ij[indices_triu], axis=1)
    qi_qj_triu = qi_qj[indices_triu]
    
    return (qi_qj_triu * 1. / r_ij_triu).sum()

@jax.jit
def calc_total_f_ref(positions, charges):
    return -jax.grad(calc_total_e_ref, argnums=0)(positions, charges)

@jax.jit
def calc_total_e_and_f_ref(positions, charges):
    value, grad =  jax.value_and_grad(calc_total_e_ref, argnums=0)(positions, charges)
    return value, -grad

In [27]:
@jax.jit
def calc_total_e_msm(positions, charges):
    return wrapped_compute_U_zero_nbl(positions, charges) + calc_longrange_e_approx(positions, charges)

@jax.jit
def calc_total_f_msm_by_grad(positions, charges):
    return -jax.grad(calc_total_e_msm, argnums=0)(positions, charges)

In [28]:
print("ref:", calc_total_e_ref(pos, chg))
print("MSM:", calc_total_e_msm(pos, chg))

ref: -15.588384775091773
MSM: -15.649109934416515


In [29]:
f_total_ref = calc_total_f_ref(pos, chg)
f_total_msm = calc_total_f_msm_by_grad(pos, chg)

In [30]:
# Compare exact and grid forces
fig, ax = plt.subplots()
ax.set_xlabel("forces (reference)")
ax.set_ylabel("forces (MSM)")
ax.scatter(f_total_ref, f_total_msm)
xlim = ax.get_xlim()
ylim = ax.get_ylim()
# plot parity line and zero axes
ax.plot(xlim, xlim, color="black", zorder=-1)
ax.axhline(color="black", linewidth=0.75)
ax.axvline(color="black", linewidth=0.75)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.show()

<IPython.core.display.Javascript object>

In [31]:
jax.device_put(pos)
jax.device_put(chg)

%timeit calc_total_e_msm(pos, chg).block_until_ready()

7.25 ms ± 350 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [32]:
jax.device_put(pos)
jax.device_put(chg)

%timeit calc_total_e_ref(pos, chg).block_until_ready()

2.41 ms ± 194 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [62]:
suggested_n_levels = suggest_max_gridlevel_nonPBC(
    min_pos=onp.zeros_like(box_lengths),
    max_pos=box_lengths,
    nb_particles=n_particles,
    level_one_gridspacing=LEVEL_ONE_GRIDSPACING,
    level_zero_cutoff=LEVEL_ZERO_CUTOFF,
    p=P,
)

Suggested value for max grid level L was found according to criterion/criteria:
(if multiple criteria listed, that means they agree on the suggested L)
*) Cutoff at highest-but-one level not too large relative to simulation box size ("criterion 2.")


## Experiment with determination of grid spacing and n_levels in periodic case

In [33]:
# h_attempt = 2.5
# 
# n_levels_including_highest = int(jnp.round(jnp.log2(side_length / h_attempt) + 1))
# h_actual = side_length / (2 ** (n_levels_including_highest - 1))
# n_levels_periodic = n_levels_including_highest - 1

In [34]:
# side_length / h_actual

In [35]:
# 2 ** (n_levels_including_highest - 1) * h_actual

In [36]:
# def get_periodic_gridspacing_and_n_levels(h, side_length):
#     level_where_only_one_point = int(jnp.round(jnp.log2(side_length / h) + 1))
#     h_actual = side_length / (2 ** (level_where_only_one_point - 1))
#     n_levels = level_where_only_one_point - 1
#     # TODO: return highest level (where only one point) or the one below?
#     return h_actual, n_levels

In [37]:
# get_periodic_gridspacing_and_n_levels(h=LEVEL_ONE_GRIDSPACING, side_length=side_length)

In [38]:
# spacings_test = [2.5, 2.0, 3.0]
# box_lengths_test = [21.0, 21.5, 32.]

In [39]:
# levels_where_only_one_point = []
# for h, length in zip(spacings_test, box_lengths_test):
#     

In [40]:
# for h, length in zip(spacings_test, box_lengths_test):
#     print(get_periodic_gridspacing_and_n_levels(h, length))

In [41]:
# lvl_ceil = int(jnp.ceil(jnp.log2(length / h) + 1))

In [42]:
# set_up_grid_axis(length=10., h=10., p=P, J_zeroplus=J_zeroplus, periodic=True)

# Clean(er)

## Global settings

In [59]:
# Basic geometry
AVG_NEIGHBOR_DISTANCE = 2.5
N_DIM = 3
PBCS = [False] * N_DIM

# MSM
LEVEL_ONE_GRIDSPACING = AVG_NEIGHBOR_DISTANCE
ALPHA = 4.0
LEVEL_ZERO_CUTOFF = ALPHA * LEVEL_ONE_GRIDSPACING
P = 4
MU = 4

## Particle setup

In [94]:
def set_up_msm(
    level_one_gridspacing,
    level_zero_cutoff,
    box_lengths,
    pbcs,
    neighborlist_reference_positions,
    n_levels=None,  # TODO: determine automatically?
    p=None,  # TODO: determine automatically?
    mu=None,  # TODO: determine automatically?
    **neighbor_kwargs,
):
    pbcs = jnp.asarray(pbcs)
    if pbcs.any():
        raise ValueError(
            "Periodic or mixed boundary conditions currently not supported."
        )

    alpha = level_zero_cutoff / level_one_gridspacing
    if p is None:
        p = suggest_p(alpha)

    # TODO: mu
    # See section "1. Preprocessing" of the article
    if mu is None:
        mu = max(int(4 * alpha + p // 2), 3 * p // 2)

    # TODO: n_levels
    if n_levels is None:
        n_levels = suggest_max_gridlevel_nonPBC(
            min_pos=onp.zeros_like(box_lengths),
            max_pos=box_lengths,
            nb_particles=neighborlist_reference_positions.shape[0],
            level_one_gridspacing=level_one_gridspacing,
            level_zero_cutoff=level_zero_cutoff,
            p=p,
        )

    kernels = split_one_over_r_kernel(
        max_level=n_levels,
        level_zero_cutoff=level_zero_cutoff,
        softening_function=SofteningFunctionOneOverR(p),
    )
    wrapped_calc_U_zero = make_wrapped_compute_U_zero(
        kernels=kernels,
        cutoff=level_zero_cutoff,
        box_lengths=box_lengths,
        pbcs=pbcs,
        neighborlist_reference_positions=neighborlist_reference_positions,
    )
    calc_U_oneplus = make_compute_U_oneplus(
        kernels=kernels,
        level_one_gridspacing=level_one_gridspacing,
        level_zero_cutoff=level_zero_cutoff,
        n_levels=n_levels,
        box_lengths=box_lengths,
        pbcs=pbcs,
        p=p,
        mu=mu,
    )

    def calculate(positions, charges):
        return wrapped_calc_U_zero(positions, charges) + calc_U_oneplus(
            positions, charges
        )

    return calculate


In [159]:
N_PARTICLES = 10000

rng = onp.random.default_rng(54)

# Randomly drawn particle positions
side_length = N_PARTICLES ** (1. / N_DIM) * AVG_NEIGHBOR_DISTANCE
box_lengths = jnp.array([side_length] * N_DIM)
pos = rng.uniform(low=onp.zeros_like(box_lengths), high=box_lengths, size=(N_PARTICLES, N_DIM))
chg = rng.uniform(low=-1.0, high=1.0, size=N_PARTICLES)
pos = jnp.array(pos)
chg = jnp.array(chg)

box_lengths

Array([53.86086725, 53.86086725, 53.86086725], dtype=float64)

In [160]:
calculate_msm_energy = set_up_msm(
    level_one_gridspacing=LEVEL_ONE_GRIDSPACING,
    level_zero_cutoff=LEVEL_ZERO_CUTOFF,
    box_lengths=box_lengths,
    pbcs=PBCS,
    neighborlist_reference_positions=pos,
)
calculate_msm_energy = jax.jit(calculate_msm_energy)

Suggested value for max grid level L was found according to criterion/criteria:
(if multiple criteria listed, that means they agree on the suggested L)
*) Highest-level grid not coarser than simulation box size ("criterion 1.")
*) Cutoff at highest-but-one level not too large relative to simulation box size ("criterion 2.")


2024-03-06 00:03:48.025850: W external/tsl/tsl/framework/bfc_allocator.cc:485] Allocator (GPU_0_bfc) ran out of memory trying to allocate 762.94MiB (rounded to 800000000)requested by op 
2024-03-06 00:03:48.028967: W external/tsl/tsl/framework/bfc_allocator.cc:497] ******************************************_______**************************__________________**_****
2024-03-06 00:03:48.029093: E external/xla/xla/pjrt/pjrt_stream_executor_client.cc:2461] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 800000000 bytes.
BufferAssignment OOM Debugging.
BufferAssignment stats:
             parameter allocation:         0B
              constant allocation:         0B
        maybe_live_out allocation:  762.94MiB
     preallocated temp allocation:         0B
                 total allocation:  762.94MiB
              total fragmentation:         0B (0.00%)
Peak buffers:
	Buffer 1:
		Size: 762.94MiB
		Operator: op_name="jit(candidate_fn)/jit(main)/broad

XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 800000000 bytes.
BufferAssignment OOM Debugging.
BufferAssignment stats:
             parameter allocation:         0B
              constant allocation:         0B
        maybe_live_out allocation:  762.94MiB
     preallocated temp allocation:         0B
                 total allocation:  762.94MiB
              total fragmentation:         0B (0.00%)
Peak buffers:
	Buffer 1:
		Size: 762.94MiB
		Operator: op_name="jit(candidate_fn)/jit(main)/broadcast_in_dim[shape=(10000, 10000) broadcast_dimensions=(1,)]" source_file="/tmp/ipykernel_331757/155299916.py" source_line=20
		XLA Label: iota
		Shape: s64[10000,10000]
		==========================



In [ ]:
print("ref:", calc_total_e_ref(pos, chg))
print("msm:", calculate_msm_energy(pos, chg))

In [ ]:
jax.device_put(pos)
jax.device_put(chg)

print("Reference:")
%timeit calc_total_e_ref(pos, chg).block_until_ready()
print()

print("MSM:")
%timeit calculate_msm_energy(pos, chg).block_until_ready()